# Part 2: Designing and Training the Chemistry Surrogate

**Learning objectives.** By the end of this notebook you will be able to:

- Choose input/output representations for a physical multi-output regression
- Build and compare MLP architectures (plain vs residual) in PyTorch
- Train on a GPU with mixed precision and interpret loss curves
- Evaluate a surrogate the way a scientist must: per-target errors,
  autoregressive rollout stability, and an honest speed benchmark

**Prerequisites:** Part 1, and a merged `surrogate_training_data.h5`
(from the SLURM job array, or a small local run of `generate_data.py`).

In [ ]:
import time
import pickle

import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
with h5py.File("surrogate_training_data.h5", "r") as f:
    X_train = f["X_train"][:]
    Y_train = f["Y_train"][:]
    X_test = f["X_test"][:]
    Y_test = f["Y_test"][:]
    feature_names = list(f.attrs["features"])
    target_names = list(f.attrs["targets"])

print(f"Train: {X_train.shape},  Test: {X_test.shape}")
print(f"Inputs : {feature_names}")
print(f"Targets: {target_names}")

## Representation choices — the part that actually matters

Architecture tweaks matter less than how you frame inputs and outputs.
Three decisions to make deliberately:

1. **Scaling.** log T, log n, and log dt live on different ranges;
   standardize them. The fractions are already in [0, 1] — leave them.
2. **Deltas vs absolutes.** For small dt the output nearly equals the input.
   Predicting the *change* (Δlog T, Δx) gives the network an easy zero
   baseline and better conditioning.
3. **Bounded outputs.** Fractions must stay in [0, 1]. We can clip after the
   fact, or build the constraint into the architecture. Try both in Ex. 2
   

In [ ]:
# Standardize the three unbounded inputs; leave fractions untouched.
CONT = [0, 1, 5]  # columns: log T, log n_H, log dt
x_mean = X_train[:, CONT].mean(axis=0)
x_std = X_train[:, CONT].std(axis=0)

def scale_X(X):
    Xs = X.copy()
    Xs[:, CONT] = (Xs[:, CONT] - x_mean) / x_std
    return Xs

# Targets as deltas: [d logT, d x_HII, d x_HeII, d x_HeIII]
# (input columns 0, 2, 3, 4 hold the corresponding initial values)
IN_STATE = [0, 2, 3, 4]
dY_train = Y_train - X_train[:, IN_STATE]
dY_test = Y_test - X_test[:, IN_STATE]

X_train_t = torch.tensor(scale_X(X_train), dtype=torch.float32).to(device)
X_test_t = torch.tensor(scale_X(X_test), dtype=torch.float32).to(device)
dY_train_t = torch.tensor(dY_train, dtype=torch.float32).to(device)
dY_test_t = torch.tensor(dY_test, dtype=torch.float32).to(device)

print("Typical |d logT| :", np.abs(dY_train[:, 0]).mean().round(3))
print("Typical |d x_HII|:", np.abs(dY_train[:, 1]).mean().round(3))

## Two architectures

We compare a plain MLP against one with residual (skip) connections.
Residual blocks let each layer learn a *correction* to an identity map,
which typically trains faster and deeper without degradation. This is the same
principle behind ResNets (He et al. 2016, CVPR). For a 6-input, 4-output
problem both are small. the point is to practice reading the comparison

In [ ]:
class PlainMLP(nn.Module):
    def __init__(self, n_in: int, n_out:int, hidden: int, n_layers: int):
        """
        n_in: Dimension of input
        n_out: Dimesnion of output
        hidden: Dimension of model hidden layers
        n_layers: Number of hidden layers
        """
        super().__init__()
        layers = [nn.Linear(n_in, hidden), nn.GELU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.GELU()]
        layers += [nn.Linear(hidden, n_out)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ResidualBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.f(x))   # skip connection


class ResidualMLP(nn.Module):
    def __init__(self, n_in: int, n_out: int, hidden: int, n_blocks: int):
        """
        n_in: Dimension of input
        n_out: Dimesnion of output
        hidden: Dimension of model hidden layers
        n_blocks: Number of residual blocks. Each residual block comprises two hidden layers.
        """
        super().__init__()
        self.inp = nn.Linear(n_in, hidden)
        self.blocks = nn.Sequential(*[ResidualBlock(hidden)
                                      for _ in range(n_blocks)])
        self.out = nn.Linear(hidden, n_out)

    def forward(self, x):
        return self.out(self.blocks(nn.functional.gelu(self.inp(x))))

In [ ]:
def train_model(model, epochs=100, batch_size=128, lr=1e-1, use_amp=False):
    """Train with AdamW + cosine schedule. Returns loss history."""
    loader = DataLoader(TensorDataset(X_train_t, dY_train_t),
                        batch_size=batch_size, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()
    scaler = torch.amp.GradScaler(enabled=use_amp)
    hist = {"train": [], "val": []}

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for xb, yb in loader:
            opt.zero_grad()
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            batch_losses.append(loss.item())
        sched.step()

        model.eval()
        with torch.no_grad():
            val = loss_fn(model(X_test_t), dY_test_t).item()
        hist["train"].append(np.mean(batch_losses))
        hist["val"].append(val)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  epoch {epoch:3d}  train {hist['train'][-1]:.5f}"
                  f"  val {val:.5f}")
    return hist

## Set up your model architecture and training procedure!


#### Things to try:
- Which model trains faster (loss decreases fastest)?
- Which model costs more to train (time per epoch)?
- Which model is more stable (easier to converge)?
    - Largest learning rate?
    - Largest batch size?
- How does training convergence depdend on model size?
    - Number of parameters?
    - Model width?
    - Model depth?
- How does feature standardization affect model size?
- Can you test these questions in an automated way?


#### Hints

When in doubt, check the class information! Try: `PlainMLP?` or `help(PlainMLP)`. If available, these commands will show any available documentation.

To see the constructed model architecture, try `print(model_plain)` or `print(model_res)`

Setting `n_in`: what is the shape of `X_train_t`?

Setting `n_out`: what is the shape of `dY_train_t`?

Setting `hidden`: Controls the model 'width' and has the biggest effect on number of model parameters ($O(N^2)$ dependence). How many parameters is enough to train your model well? Should be larger than `n_in` and `n_out`.

Setting `n_layers`/`n_blocks`: Controls the model 'depth' and has moderate effect on number of model parameters ($O(N)$ dependence). 

Generally, for training: 
- for model architecture parameters: start small and slowly increase until model begins to converge stably. Sizes that are powers of 2 tend to be hardware-friendly.
- for learning rate: make as large as possible, i.e., largest value where training can still converge consistently
- for batch_size: Depends on data set size and model size. Choose values which make training faster, and use other parameters to control model quality.
- Start with small, fast experiments; get a feel for how things play out; scale up and verify!

### Set up the model

In [ ]:
torch.manual_seed(42)

#### STUDENT FILLS OUT
n_in: int = None         # Number of features in (dimension/rank of input)
n_out: int = None        # Number of features out (dimension/rank of output)
hidden_dim: int = None   # Hidden dimension of the multi-layer perceptron/fully-connected neural network
n_layers: int = None     # Number of layers in the standard MLP
n_blocks: int = None     # Number of blocks in the Residual MLP

shared_architecture = {
    'n_in': n_in,
    'n_out': n_out, 
    'hidden': hidden_dim, 
}

plain_arch = {
    'n_layers': n_layers,
    **shared_architecture,
}

residual_arch = {
    'n_blocks': n_blocks,
    **shared_architecture,
}

#### STUDENT FILLS OUT

model_plain = PlainMLP(**plain_arch).to(device)
model_res = ResidualMLP(**residual_arch).to(device)

print(f"PlainMLP params:    {sum(p.numel() for p in model_plain.parameters()):,}")
print(f"ResidualMLP params: {sum(p.numel() for p in model_res.parameters()):,}")

### Run the model training

In [ ]:
#### STUDENT FILLS OUT
training_parameters = {
    'epochs':100,      # Number of passes over the full dataset
    'batch_size':128,  # Number of training examples over which model update is calculated (gradient direction + size)
    'lr':1e-1,         # Learning rate. How large (relatively) of a step to take in direction of gradient (update = lr * gradient)
}
#### STUDENT FILLS OUT

print("\nTraining PlainMLP...")
hist_plain = train_model(model_plain, **training_parameters)
print("\nTraining ResidualMLP...")
hist_res = train_model(model_res, **training_parameters)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.semilogy(hist_plain["val"], label="PlainMLP (val)", color="steelblue")
ax.semilogy(hist_res["val"], label="ResidualMLP (val)", color="darkorange")
ax.semilogy(hist_plain["train"], color="steelblue", alpha=0.35, ls="--")
ax.semilogy(hist_res["train"], color="darkorange", alpha=0.35, ls="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Architecture comparison (dashed = train)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### STUDENT FILLS OUT
# carry the better model forward (check your own curves!)
def choose_model(
    plain_data: dict[str, list[float]],
    residual_data: dict[str, list[float]],
    ) -> bool:
    """Choose between the trained plain and residual MLPs, based off of the respective training curves.
    Returns True if residual MLP is chosen.
    """

    raise NotImplementedError('Use the data from hist_plain and hist_res to make an informed decision!')

# either set explicitly to True or False, or, implement the choose_model function above
# What is the decision criteria you are using?
try:
    use_resnet = choose_model(hist_plain, hist_res)
except NotImplementedError:
    use_resnet = None # set explicitly here!

#### STUDENT FILLS OUT

if use_resnet is True:
    model = model_res  
elif use_resnet is False:
    model = model_plain
else:
    raise NotImplementedError("Need to choose which model to continue with!")

## Per-target evaluation

A single MSE hides nuance under one number. We report errors in physical units, per target: dex for temperature (astronomers' unit of
log₁₀ error) and absolute error for the fractions. Ask of each number: is this smaller than the error the simulation makes anyway by using approximate chemistry? That, not the loss value, is the acceptance test.

In [ ]:
model.eval()
with torch.no_grad():
    dY_pred = model(X_test_t).cpu().numpy()

Y_pred = X_test[:, IN_STATE] + dY_pred          # back to absolute state
Y_pred[:, 1:] = np.clip(Y_pred[:, 1:], 0.0, 1.0)  # enforce bounds

print("Per-target test errors:")
labels = ["log10 T'  [dex]", "x_HII'", "x_HeII'", "x_HeIII'"]
for j, lab in enumerate(labels):
    err = Y_pred[:, j] - Y_test[:, j]
    print(f"  {lab:18s}  RMSE {np.sqrt((err**2).mean()):.4f}"
          f"   max {np.abs(err).max():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for j, (ax, lab) in enumerate(zip(axes, labels)):
    ax.scatter(Y_test[:, j], Y_pred[:, j], s=2, alpha=0.3, rasterized=True)
    lims = [Y_test[:, j].min(), Y_test[:, j].max()]
    ax.plot(lims, lims, "r--", lw=1)
    ax.set_xlabel(f"true {lab}")
    if j == 0:
        ax.set_ylabel("predicted")
    ax.set_title(lab, fontsize=10)
    ax.grid(True, alpha=0.3)
plt.suptitle("Parity plots per target")
plt.tight_layout()
plt.show()

## The rollout test — where surrogates go to die

In a simulation the surrogate is called *autoregressively*: its output becomes its next input, and errors compound. A model with beautiful
one-step parity plots can still drift or oscillate over a long rollout. We cool a hot cell for 600 Myr in 10 Myr steps, once with the true solver
and once with the network feeding itself, and compare the tracks. This will inform us whether the model will actually work in production. 

#### Things to try
- Does model training/validation error relate to how errors compound in an autoregressive rollout?
- How does model size impact rollout stability?
- At similar levels of training accuracy, are plain and residual MLPs similarly stable?
- How many steps can be run before the surrogate model gets off track?
- Is the surrogate model similarly unstable at all initial temperatures?
- Can the model be made more stable by increasing the floating point precision?

In [ ]:
import hhe_chemistry as chem

#### STUDENT FILLS OUT
n_steps: int = 60    # number of rollout steps
rollout_T: float = 1e7 # initial temperature in K
#### STUDENT FILLS OUT



Myr = 3.156e13
n_H, dt_myr = 1e-2, 10.0
log_dt = np.log10(dt_myr * Myr)

# Ground truth track
y = chem.equilibrium_state(rollout_T, n_H)
truth = [y[3]]
for _ in range(n_steps):
    y, _ = chem.integrate_cell(y, n_H, dt_myr * Myr)
    truth.append(y[3])

# Neural network rollout: feed predictions back in
state = chem.equilibrium_state(rollout_T, n_H)   # [x_HII, x_HeII, x_HeIII, T]

nn_track = [state[3]]
for _ in range(n_steps):
    x = np.array([[np.log10(state[3]), np.log10(n_H),
                   state[0], state[1], state[2], log_dt]])
    with torch.no_grad():
        d = model(torch.tensor(scale_X(x), dtype=torch.float32)
                  .to(device)).cpu().numpy()[0]
    new_logT = np.log10(state[3]) + d[0]
    state = [np.clip(state[0] + d[1], 0, 1),
             np.clip(state[1] + d[2], 0, 1),
             np.clip(state[2] + d[3], 0, 1),
             10 ** new_logT]
    nn_track.append(state[3])

t_axis = np.arange(n_steps + 1) * dt_myr

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(t_axis, truth, "k-", lw=2, label="Stiff ODE solver (truth)")
ax.semilogy(t_axis, nn_track, "darkorange", lw=2, ls="--",
            label="NN surrogate rollout")
ax.set_xlabel("Time [Myr]")
ax.set_ylabel("Temperature [K]")
ax.set_title("Autoregressive rollout: does the surrogate stay on track?")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## An honest benchmark

Rules for timing that you should carry into your own research: warm up the GPU first, synchronize before reading the clock (`torch.cuda.synchronize`),
report the median of repeats, and compare *equal work* — here, advancing the same 10,000 cells by the same dt. The solver pays per cell; the network pays once per batch. That asymmetry is what we are using to our advantage here.

### Precisely measuring GPU execution timings
Operations performed on the GPU are technically executed *asynchronously* from operations executing on the CPU. The operations can happen simultaneously, or, depending on how many operations have been scheduled to run on the GPU, could in reality drift far apart from each other. This makes timing GPU execution a bit more involved than timing the analagous CPU operations. For more info, read the hidden note!

#### Details on GPU kernel scheduling (extra reading)

When GPU operations are scheduled to run on CUDA devices, the operations themselves are scheduled within the context of a CUDA stream. These streams are essentially first-in first-out (FIFO) queues---the operations execute in the order in which they are scheduled. To time operations, we can use CUDA stream events, which are essentially no-op markers that we can use to track the stream progress and measure elapsed time. Before entering the region we want to profile, we record an event; after leaving, we record another event; and finally, we synchronize the CUDA stream so that all enqueued operations complete and use the events to determine how much time has elapsed. Below, we use the `GPUTimer` class as a context manager, which helps us perform all of these steps in the right order for the region of interest.

(A quick note: by default, when using pytorch, all operations are scheduled onto the default device stream, but you can have multiple streams per GPU!)

In [ ]:
class GPUTimer:

    def __init__(self, reporting: bool, region_name: str):
        self.reporting = reporting
        self.region_name = region_name
        self.start_event = torch.cuda.Event(enable_timing=True)
        self.end_event = torch.cuda.Event(enable_timing=True)

    def __enter__(self):
        self.start_event.record()

    def __exit__(self, *args, **kwargs):
        self.end_event.record()
        torch.cuda.synchronize()
        if self.reporting:
            self.report()
        
    def report(self):
        print(f'[{self.region_name}] took {self.elapsed_time():0.3f}s')
        
    def elapsed_time(self) -> float:
        # CUDA events record elapsed time in milliseconds
        return self.start_event.elapsed_time(self.end_event) / 1.0e3

#### Things to try

When measuring the execution timing of code, we often need to run a warm up round (or few) to get accurate timings, and if the procedure is especially fast, run many samples.
This is especially pertinent for GPU codes, for a variety of reasons. 

- How many evaluations do you need to run before the neural network surrogate timing converges? (What is its variance?)
- Do the measured timings change signficantly with(out) the warm up round?
- How about with a freshly started jupyter notebook (save the model weights and reload!)?

In [ ]:
N_BENCH = 10_000
rng = np.random.default_rng(3)
idx = rng.integers(0, len(X_test), N_BENCH)
X_bench = X_test[idx]

# --- Stiff solver timing (per-cell loop, like a simulation would do).
# We are timing a subsample and scaling up, so the cell runs in seconds not minutes.
N_SUB = 200
t0 = time.perf_counter()
for i in range(N_SUB):
    y0 = [X_bench[i, 2], X_bench[i, 3], X_bench[i, 4], 10 ** X_bench[i, 0]]
    chem.integrate_cell(y0, 10 ** X_bench[i, 1], 10 ** X_bench[i, 5])
t_solver = (time.perf_counter() - t0) / N_SUB * N_BENCH

# --- NN timing (batched, with proper GPU sync).
model.eval()
X_bench_t = torch.tensor(scale_X(X_bench), dtype=torch.float32).to(device)
with torch.no_grad():
    _ = model(X_bench_t)                       # warm-up
if device.type == "cuda":
    torch.cuda.synchronize()

reps = []
t0 = time.perf_counter()
timer = GPUTimer(False, 'Surrogate throughput benchmark')
with torch.no_grad():
    for _ in range(20):
        with timer:
            _ = model(X_bench_t)
        reps.append(timer.elapsed_time())
t_nn = np.median(reps)

print(f"Advancing {N_BENCH:,} cells by one chemistry step:")
print(f"  stiff ODE solver : {t_solver:8.2f} s   (extrapolated from {N_SUB})")
print(f"  NN surrogate     : {t_nn*1e3:8.2f} ms  on {device.type.upper()}")
print(f"  speedup          : {t_solver / t_nn:,.0f}x")

### Exercise 2 — bounded outputs by construction

Our model can predict fractions outside [0, 1]; we clip after the fact.
Modify `ResidualMLP` so the three fraction outputs are produced by a
sigmoid — i.e., predict the new fractions directly (not deltas) through
`torch.sigmoid`, while keeping Δlog T as a linear output. Retrain and
compare per-target errors and the rollout. Is building the constraint in
better than clipping, here? (There is no universally right answer — the
point is to find out empirically.)



In [ ]:
#### STUDENT FILLS OUT
# Here, we create a new model which is a subclass of the ResidualMLP model.
# Subclasses inherit methods and properties of their parent class.
# So, if you want to only modify a small portion of behavior, using
# classes and inheritance is a good, clean way to customize and reuse code.

class BoundedResidualMLP(ResidualMLP):    
    def forward(self, x):
        original_output = super().forward(x)
        raise NotImplementedError

#### STUDENT FILLS OUT

# Copy the previously used residual architecture.
# If desired, set new architecture parameters here. 
bounded_arch = {**residual_arch} 
bounded_model = BoundedResidualMLP(**bounded_arch)
hist_bounded = train_model(bounded_model)


#### STUDENT FILLS OUT
# Visualize your results!

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### STUDENT FILLS OUT

### Exercise 3 — loss weighting

The Δlog T target has larger typical magnitude than the fraction deltas,
so plain MSE prioritizes temperature. Reweight the loss so each target
contributes more equally. Does the rollout improve?


#### Things to try
- How are you determining the weights?
    - Try dividing each target's squared error by its variance in the training set.

#### Hints



- What shape should the weighting tensor be?
- The variance of a tensor can be calculated as `X.var(dim=k)`, where `k` is the dimension over which the variance is calculated. Try `torch.var?` or `help(torch.var)` for more details. 
- The mean-squared error loss (`nn.MSELoss()`) can be explicitly implemented as `((pred-target)**2).mean()`. Where should the weights go?

## Run training

In [ ]:
#### STUDENT FILLS OUT
def weighted_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    

    raise NotImplementedError
#### STUDENT FILLS OUT
    

# No need to update this, unless you change the signature of the weighted_loss function
def train_model_with_weighting(model, epochs=100, batch_size=1024, lr=1e-3, use_amp=False):
    """Train with AdamW + cosine schedule. Returns loss history."""
    loader = DataLoader(TensorDataset(X_train_t, dY_train_t),
                        batch_size=batch_size, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = weighted_loss
    scaler = torch.amp.GradScaler(enabled=use_amp)
    hist = {"train": [], "val": []}

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for xb, yb in loader:
            opt.zero_grad() 
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                loss = loss_fn(model(xb), yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            batch_losses.append(loss.item())
        sched.step()

        model.eval()
        with torch.no_grad():
            val = loss_fn(model(X_test_t), dY_test_t).item()
        hist["train"].append(np.mean(batch_losses))
        hist["val"].append(val)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  epoch {epoch:3d}  train {hist['train'][-1]:.5f}"
                  f"  val {val:.5f}")
    return hist

model_res_with_weighting = ResidualMLP(**residual_arch).to(device)

hist_weighted = train_model_with_weighting(model_res_with_weighting)

## Save the model for Part 3

Part 3 scales training across multiple GPUs with SLURM. Save the scalers so
every rank preprocesses identically.

In [ ]:
torch.save(model.state_dict(), "surrogate_model.pt")
with open("surrogate_scalers.pkl", "wb") as f:
    pickle.dump({"x_mean": x_mean, "x_std": x_std, "CONT": CONT,
                 "IN_STATE": IN_STATE}, f)
print("Saved surrogate_model.pt and surrogate_scalers.pkl")